# Comparing two arms of a trial from its patient-level file

**Data kit lab** `ist-aspirin-14-day-deaths` · Clinical · about 40 minutes

**The question.** In the International Stroke Trial, did aspirin started within 48 hours of a suspected acute ischaemic stroke reduce deaths in the first 14 days?

**What you do.** You open the trial's public file, one row per patient, and work towards one comparison.

1. Read the file and compare its header with the data dictionary.
2. Compare the corrected file with the first version.
3. Describe who was randomised.
4. Tabulate the final diagnosis.
5. Look at the allocation. The trial is factorial: aspirin or not, and heparin or not.
6. Count deaths within 14 days by aspirin allocation. Compute the percentages and the difference per 1000 patients.
7. Do the same for heparin.
8. Check your numbers against the ones the trial's authors printed.

**The data.** International Stroke Trial database (version 2), University of Edinburgh DataShare, https://datashare.ed.ac.uk/handle/10283/128. The files are downloaded from that record when you run the first cell. Nothing is stored with this notebook.

**Notice.** Contains information from the International Stroke Trial database (version 2), https://datashare.ed.ac.uk/handle/10283/128, which is made available under the ODC Attribution License. Source of the dataset: the International Stroke Trial Collaborative Group; Sandercock PAG, Niewada M, Czlonkowska A. The International Stroke Trial database. Trials 2011;12:101. The data paper says the trial was principally funded by the UK Medical Research Council, the UK Stroke Association and the European Union BIOMED-1 program.

**Published values.** Two documents by the trial's authors: the data paper (Table 4 and its Results text) and the abstract of the trial's 1997 report in the Lancet (PubMed record 9174558). The 1997 article itself is not open access, so only its abstract is used.

In [ ]:
# datakit:setup (generated from lab.yaml and expected.json by scripts/datakit/harness.py sync; do not edit by hand)
import hashlib, json, os, pathlib, urllib.request

LAB = 'ist-aspirin-14-day-deaths'
FILES = json.loads(r'''{
 "ist": {
  "url": "https://datashare.ed.ac.uk/bitstreams/eca52b56-6d1b-47b3-814a-0b7089f8c870/download",
  "name": "ist",
  "sha256": "a350aed2329bbcde23c05caaa96b9275b44b017adecc2fd02231d590454695d5",
  "bytes": 4798288
 },
 "ist-v1": {
  "url": "https://datashare.ed.ac.uk/bitstreams/446b7697-5025-46df-ab3b-bc3933e4bc70/download",
  "name": "ist-v1",
  "sha256": "ea525e3ba13061d1a955c1956c100a3cfe16b4f4b522ecad4dd15ea6cdb80af7",
  "bytes": 4827869
 },
 "dictionary": {
  "url": "https://www.ebi.ac.uk/europepmc/webservices/rest/PMC3104487/fullTextXML",
  "name": "dictionary",
  "sha256": "c0a0230b3060d554e9c7e94f87f88d56270bbe055eaf4d37b3875f64c4478d93",
  "bytes": 66421
 },
 "erratum": {
  "url": "https://www.ebi.ac.uk/europepmc/webservices/rest/PMC3320519/fullTextXML",
  "name": "erratum",
  "sha256": "a68cf66a9c54b6204070ef6a457aefaba2c71acd3a18e1eeb2b1547b9810fe09",
  "bytes": 9789
 }
}''')
PUBLISHED = json.loads(r'''{
 "n_total": {
  "label": "Patients randomised",
  "value": 19435,
  "tolerance": {
   "exact": true
  }
 },
 "n_dx_ischaemic": {
  "label": "Final diagnosis: ischaemic stroke",
  "value": 17398,
  "tolerance": {
   "exact": true
  }
 },
 "n_dx_haemorrhagic": {
  "label": "Final diagnosis: haemorrhagic stroke",
  "value": 599,
  "tolerance": {
   "exact": true
  }
 },
 "n_dx_unknown_type": {
  "label": "Final diagnosis: stroke, type unknown",
  "value": 992,
  "tolerance": {
   "exact": true
  }
 },
 "n_dx_not_stroke": {
  "label": "Final diagnosis: not a stroke",
  "value": 420,
  "tolerance": {
   "exact": true
  }
 },
 "n_dx_uncertain": {
  "label": "Final diagnosis: uncertain",
  "value": 26,
  "tolerance": {
   "exact": true
  }
 },
 "n_hospitals": {
  "label": "Hospitals",
  "value": 467,
  "tolerance": {
   "exact": true
  }
 },
 "n_countries": {
  "label": "Countries",
  "value": 36,
  "tolerance": {
   "exact": true
  }
 },
 "n_ct_before": {
  "label": "CT scan before randomisation",
  "value": 13020,
  "tolerance": {
   "exact": true
  }
 },
 "n_over_80": {
  "label": "Aged over 80 at entry",
  "value": 5132,
  "tolerance": {
   "exact": true
  }
 },
 "pct_over_80": {
  "label": "Aged over 80 at entry, percent of all randomised",
  "value": 26.4,
  "tolerance": {
   "abs": 0.05
  }
 },
 "deaths14_aspirin": {
  "label": "Deaths within 14 days, allocated aspirin",
  "value": 872,
  "tolerance": {
   "exact": true
  }
 },
 "deaths14_no_aspirin": {
  "label": "Deaths within 14 days, allocated no aspirin",
  "value": 909,
  "tolerance": {
   "exact": true
  }
 },
 "pct_deaths14_aspirin": {
  "label": "Deaths within 14 days, percent of those allocated aspirin",
  "value": 9.0,
  "tolerance": {
   "abs": 0.05
  }
 },
 "pct_deaths14_no_aspirin": {
  "label": "Deaths within 14 days, percent of those allocated no aspirin",
  "value": 9.4,
  "tolerance": {
   "abs": 0.05
  }
 },
 "fewer_per_1000_aspirin": {
  "label": "Fewer deaths within 14 days per 1000 patients, aspirin against no aspirin",
  "value": 4.0,
  "tolerance": {
   "abs": 0.5
  }
 },
 "deaths14_heparin": {
  "label": "Deaths within 14 days, allocated heparin",
  "value": 876,
  "tolerance": {
   "exact": true
  }
 },
 "deaths14_no_heparin": {
  "label": "Deaths within 14 days, allocated no heparin",
  "value": 905,
  "tolerance": {
   "exact": true
  }
 },
 "pct_deaths14_heparin": {
  "label": "Deaths within 14 days, percent of those allocated heparin",
  "value": 9.0,
  "tolerance": {
   "abs": 0.05
  }
 },
 "pct_deaths14_no_heparin": {
  "label": "Deaths within 14 days, percent of those allocated no heparin",
  "value": 9.3,
  "tolerance": {
   "abs": 0.05
  }
 },
 "fewer_per_1000_heparin": {
  "label": "Fewer deaths within 14 days per 1000 patients, heparin against no heparin",
  "value": 3.0,
  "tolerance": {
   "abs": 0.5
  }
 }
}''')
PUBLISHED_FROM = "Two documents by the trial's authors. (A) Europe PMC full-text record PMC3104487 (XML, no page numbers): Table 4, Final diagnosis of initial event, and the Results text: eleven checks. (B) PubMed record PMID 9174558, abstract of the 1997 Lancet report, section RESULTS: ten checks, deaths within 14 days by allocated treatment. Each check names its document. | https://www.ebi.ac.uk/europepmc/webservices/rest/PMC3104487/fullTextXML"
DATA_DIR = pathlib.Path(os.environ.get('DATAKIT_DATA', 'datakit-data/' + LAB))


def data_file(file_id):
    """Path of one dataset file. Downloads it from its source on first use and checks the checksum."""
    f = FILES[file_id]
    path = DATA_DIR / f['name']
    if not path.exists():
        DATA_DIR.mkdir(parents=True, exist_ok=True)
        print('downloading', f['url'])
        urllib.request.urlretrieve(f['url'], path)
    if f['sha256']:
        got = hashlib.sha256(path.read_bytes()).hexdigest()
        if got != f['sha256']:
            print('WARNING:', path.name, 'differs from the copy this lab was checked against (the source file changed)')
    return path


def within(computed, check):
    tol, pub = check['tolerance'], check['value']
    if 'exact' in tol:
        return computed == pub
    limit = tol['abs'] if 'abs' in tol else tol['rel'] * abs(pub)
    return abs(computed - pub) <= limit + 1e-12


def report(results):
    """Print computed against published values and save the computed values for the harness."""
    print('Published values:', PUBLISHED_FROM)
    print(f"{'check':<44}{'published':>12}{'computed':>14}  result")
    for key, check in PUBLISHED.items():
        got = results.get(key)
        shown = '(missing)' if got is None else (f'{got:.4f}' if isinstance(got, float) else str(got))
        verdict = 'missing' if got is None else ('match' if within(got, check) else 'DIFFERENT')
        print(f"{check['label'][:43]:<44}{str(check['value']):>12}{shown:>14}  {verdict}")
    out = pathlib.Path(os.environ.get('DATAKIT_RESULTS', 'results.json'))
    out.write_text(json.dumps(results, indent=1))
    print('saved', out)


## 1. Read the file

The file is a CSV with one row per randomised patient. Try the default encoding first and see what happens.

In [ ]:
import io
import math
import re
import xml.etree.ElementTree as ET

import pandas as pd

path = data_file("ist")
try:
    pd.read_csv(path, low_memory=False)
    print("read with the default encoding (UTF-8)")
except UnicodeDecodeError as err:
    print("the default encoding fails:", err.reason, "at byte", err.start)

ist = pd.read_csv(path, encoding="latin-1", low_memory=False)
print("rows:", len(ist), " columns:", ist.shape[1])
print(ist[["SEX", "AGE", "RXASP", "RXHEP", "ID14"]].head().to_string())

The file is not valid UTF-8. Some free-text comment columns hold other bytes. Latin-1 reads every byte.

## 2. Header against dictionary

The data dictionary is Table 2 of the data paper. Read the variable names from the paper's XML and compare them with the file's header.

In [ ]:
def text_of(el):
    return " ".join("".join(el.itertext()).split())


paper = ET.fromstring(data_file("dictionary").read_bytes())
table2 = next(tw for tw in paper.iter("table-wrap") if tw.get("id") == "T2")
rows = [[text_of(c) for c in tr if c.tag in ("td", "th")] for tr in table2.iter("tr")]
dictionary = {r[0]: r[1] for r in rows if len(r) == 2}
print("variables in the dictionary:", len(dictionary))
print("in the file, not in the dictionary:", [c for c in ist.columns if c not in dictionary])
print("in the dictionary, not in the file:", [v for v in dictionary if v not in ist.columns])
for name in ["RXASP", "RXHEP", "ID14", "SET14D", "DDEAD"]:
    print(f"{name}: {dictionary[name][:150]}")

One name differs. The dictionary calls the death indicator for the whole follow-up `ID`. The file calls it `DIED`. This lab uses `ID14`, the indicator of death at 14 days, which has the same name in both.

Read the comment on `RXHEP`. The higher heparin dose was coded `H` in the pilot phase and `M` in the main trial.

## 3. Two versions of the file

The authors published an erratum. It says errors were introduced into date variables when the file was anonymised, and that a corrected file is at the Edinburgh repository. The repository keeps the first version too. Compare the two, column by column.

In [ ]:
v1 = pd.read_csv(data_file("ist-v1"), encoding="latin-1", low_memory=False)
padded = [c for c in v1.columns if c != c.strip()]
print("first version: rows", len(v1), "columns", v1.shape[1], " padded column names:", padded)
v1.columns = [c.strip() for c in v1.columns]
assert list(v1.columns) == list(ist.columns), "the two versions have different columns"

differ = [c for c in ist.columns if not (v1[c].fillna("").astype(str).values == ist[c].fillna("").astype(str).values).all()]
print("columns that differ between the versions:", len(differ))

erratum = ET.fromstring(data_file("erratum").read_bytes())
listed = re.findall(r"([A-Z][A-Z0-9_]+): Date", text_of(erratum.find(".//body")))
print("date variables listed in the erratum:", len(listed))
print("  differ and listed:", [c for c in differ if c in listed])
print("  differ, not listed:", [c for c in differ if c not in listed])
print("  listed, do not differ:", [c for c in listed if c not in differ])
for c in [c for c in differ if c not in listed]:
    print(f"  {c}: first version {list(v1[c].head(3))}, corrected {list(ist[c].head(3))}")

USED = ["HOSPNUM", "CNTRYNUM", "RCT", "AGE", "DDIAGISC", "DDIAGHA", "DDIAGUN", "DNOSTRK", "RXASP", "RXHEP", "ID14", "SET14D"]
print("columns this lab uses that differ between versions:", [c for c in USED if c in differ])

The comparison is by text, cell by cell. Twelve of the thirteen date variables the erratum lists differ between the versions. `DPED` is listed and does not differ. `RDATE`, the month of randomisation, is not listed and differs in every row: the first version writes `1991-01` and the corrected file writes `sty-91`, a short month name that is not English. The lab does not establish why. If you need `RDATE`, parse it with care.

None of the columns used below changed, so the numbers in this lab are the same in both versions.

Record which version you analyse. From here on it is the corrected file.

## 4. Who was randomised

In [ ]:
n_total = len(ist)
n_hospitals = ist["HOSPNUM"].nunique()
n_countries = ist["CNTRYNUM"].nunique()
n_ct_before = int((ist["RCT"] == "Y").sum())
n_over_80 = int((ist["AGE"] > 80).sum())
pct_over_80 = 100 * n_over_80 / n_total
print("patients randomised:", n_total)
print("hospitals:", n_hospitals, " countries:", n_countries)
print("CT scan before randomisation:", n_ct_before)
print(f"aged over 80: {n_over_80} ({pct_over_80:.1f}%)")

## 5. Final diagnosis of the initial event

Patients were randomised on a clinical diagnosis of ischaemic stroke. The treating doctor recorded a final diagnosis later, on the 14-day form. Four fields hold it. Look at the codes before you count.

In [ ]:
DX = {"DDIAGISC": "Ischaemic stroke", "DDIAGHA": "Haemorrhagic stroke", "DDIAGUN": "Stroke, type unknown", "DNOSTRK": "Not a stroke"}
for col in DX:
    print(col, ist[col].value_counts(dropna=False).to_dict())

is_yes = ist[list(DX)].eq("Y")
print("patients with more than one Y:", int((is_yes.sum(axis=1) > 1).sum()))
dx_counts = {col: int(is_yes[col].sum()) for col in DX}
n_dx_uncertain = int((is_yes.sum(axis=1) == 0).sum())
for col, label in DX.items():
    print(f"{label:<24}{dx_counts[col]:>7}")
print(f"{'Uncertain (no Y)':<24}{n_dx_uncertain:>7}")
print(f"{'Total':<24}{sum(dx_counts.values()) + n_dx_uncertain:>7}")

The fields hold `Y`, `N`, `U`, a lower-case `u` and empty cells. Counting `Y` gives one diagnosis per patient at most. The rest are uncertain.

Not everyone randomised had an ischaemic stroke in the end. The published comparison still covers everyone randomised. The final diagnosis was recorded after randomisation, so it cannot define who is in the comparison.

## 6. Allocation

Each patient was allocated aspirin or no aspirin, and one of two heparin doses or no heparin. Count the cells.

In [ ]:
print(pd.crosstab(ist["RXASP"], ist["RXHEP"], margins=True).to_string())
ist["aspirin"] = ist["RXASP"] == "Y"
ist["heparin"] = ist["RXHEP"].isin(["L", "M", "H"])
assert ist["RXHEP"].isin(["L", "M", "H", "N"]).all() and ist["RXASP"].isin(["Y", "N"]).all()
print()
print(pd.crosstab(ist["aspirin"], ist["heparin"], margins=True).to_string())

Heparin has four codes. `H` and `M` are the same dose under two names. `N` is "avoid heparin".

Half the patients were allocated aspirin and half were allocated heparin, independently. So the same file gives two comparisons: aspirin against no aspirin, and heparin against no heparin.

## 7. Deaths within 14 days, by aspirin allocation

`ID14` is 1 if the patient died within 14 days. Compare by allocated treatment, with every allocated patient in the denominator.

In [ ]:
def compare(data, arm):
    """Deaths within 14 days by allocation to `arm` (a True/False column). Everyone allocated is in the denominator."""
    yes, no = data[data[arm]], data[~data[arm]]
    n1, n0 = len(yes), len(no)
    d1, d0 = int(yes["ID14"].sum()), int(no["ID14"].sum())
    p1, p0 = d1 / n1, d0 / n0
    se = math.sqrt(p1 * (1 - p1) / n1 + p0 * (1 - p0) / n0)
    return {"n": n1, "n_no": n0, "deaths": d1, "deaths_no": d0, "pct": 100 * p1, "pct_no": 100 * p0,
            "fewer_per_1000": 1000 * (p0 - p1), "se_per_1000": 1000 * se}


def show(result, arm):
    print(f"allocated {arm:<10} {result['deaths']:>5} of {result['n']:>5} died within 14 days ({result['pct']:.2f}%)")
    print(f"allocated no {arm:<7} {result['deaths_no']:>5} of {result['n_no']:>5} died within 14 days ({result['pct_no']:.2f}%)")
    low = result["fewer_per_1000"] - 1.96 * result["se_per_1000"]
    high = result["fewer_per_1000"] + 1.96 * result["se_per_1000"]
    print(f"fewer deaths per 1000 patients with {arm}: {result['fewer_per_1000']:.2f} (95% interval {low:.1f} to {high:.1f}, normal approximation)")


aspirin = compare(ist, "aspirin")
show(aspirin, "aspirin")

print()
print("14-day status known (SET14D):", ist["SET14D"].value_counts().to_dict())
known = compare(ist[ist["SET14D"] == 1], "aspirin")
print(f"with only patients of known status in the denominator: {known['pct']:.2f}% against {known['pct_no']:.2f}%")

The interval is not in the published abstract. It is shown so you can see how wide it is. It includes zero.

Two patients have unknown status at 14 days. Leaving them out does not change the percentages at the printed precision.

## 8. Deaths within 14 days, by heparin allocation

In [ ]:
heparin = compare(ist, "heparin")
show(heparin, "heparin")

## 9. The four cells

Shown, not checked: the same outcome in each of the four combinations of aspirin and heparin.

In [ ]:
cells4 = ist.groupby(["aspirin", "heparin"])["ID14"].agg(patients="size", deaths="sum")
cells4["percent"] = (100 * cells4["deaths"] / cells4["patients"]).round(2)
print(cells4.to_string())

## 10. Check against the published values

In [ ]:
RESULTS = {
    "n_total": n_total,
    "n_dx_ischaemic": dx_counts["DDIAGISC"],
    "n_dx_haemorrhagic": dx_counts["DDIAGHA"],
    "n_dx_unknown_type": dx_counts["DDIAGUN"],
    "n_dx_not_stroke": dx_counts["DNOSTRK"],
    "n_dx_uncertain": n_dx_uncertain,
    "n_hospitals": n_hospitals,
    "n_countries": n_countries,
    "n_ct_before": n_ct_before,
    "n_over_80": n_over_80,
    "pct_over_80": pct_over_80,
}
for arm, result in [("aspirin", aspirin), ("heparin", heparin)]:
    RESULTS[f"deaths14_{arm}"] = result["deaths"]
    RESULTS[f"deaths14_no_{arm}"] = result["deaths_no"]
    RESULTS[f"pct_deaths14_{arm}"] = result["pct"]
    RESULTS[f"pct_deaths14_no_{arm}"] = result["pct_no"]
    RESULTS[f"fewer_per_1000_{arm}"] = result["fewer_per_1000"]
report(RESULTS)

## 11. What does not reproduce

Every count and percentage of deaths within 14 days matches the abstract of the trial report, and so does the final-diagnosis table. Some descriptive counts in the text of the data paper do not match the file. The lab keeps them as they are. The next cell lists them and looks at each one.

In [ ]:
different = [key for key, check in PUBLISHED.items() if not within(RESULTS[key], check)]
print("checks that differ:", different)

print()
print("Age. The data paper says age in the public file was rounded to the nearest whole year.")
strictly, at_least = int((ist["AGE"] > 80).sum()), int((ist["AGE"] >= 80).sum())
print(f"  AGE > 80: {strictly}   AGE >= 80: {at_least}   AGE == 80: {at_least - strictly}")
published = PUBLISHED["n_over_80"]["value"]
print(f"  the published count ({published}) lies between the two: {strictly < published < at_least}")

print()
print("CT before randomisation, against infarct visible on CT:")
print(pd.crosstab(ist["RCT"], ist["RVISINF"], margins=True).to_string())
print("  cells of RVISINF equal to N:", int((ist["RVISINF"] == "N").sum()), " published count of CT before randomisation:", PUBLISHED["n_ct_before"]["value"])

print()
print("Hospitals. Distinct HOSPNUM:", ist["HOSPNUM"].nunique(), " lowest:", ist["HOSPNUM"].min(), " highest:", ist["HOSPNUM"].max())

What can be said, and no more:

- **Age.** The published count of patients over 80 lies between the count of rounded ages above 80 and the count of rounded ages of 80 and above. That is what you would see if the published count was made on exact ages before rounding. The lab cannot confirm it, because the public file holds only rounded ages.
- **CT before randomisation.** The file has a few more patients coded `Y` than the paper states. The number of `N` cells in the neighbouring column `RVISINF` is the same as the paper's count. This may be a coincidence. The cause is not established.
- **Hospitals.** The file has one fewer distinct hospital code than the paper states. The cause is not established.

A published file and the text that describes it are made at different times and sometimes from different copies of the data. Small differences in descriptive counts are common. The honest report is the one above: which numbers match, which do not, and what is known about why.

## What the result says

Look at the two percentages for aspirin and the difference per 1000 patients. Then look at the interval in section 7. Is the difference in deaths within 14 days clear on its own? Read the abstract of the 1997 report (PubMed 9174558): it calls the difference non-significant and bases its conclusion for aspirin on other outcomes as well, recurrent stroke within 14 days and death or dependency at six months.

## Try next

1. The abstract also gives recurrent ischaemic stroke within 14 days by aspirin allocation. The file has an indicator `ISC14`. Compute the two percentages and compare them with the abstract yourself. Which denominator did you use?
2. Repeat section 7 for patients whose final diagnosis was ischaemic stroke. Why is this not the trial's result?
3. The file has `TD` (days to death or censoring) and `DIED`. Plot the proportion still alive over six months by aspirin allocation. No published number for that curve is checked in this lab.
4. Compare the two heparin doses (`L` against `M` and `H`) for deaths within 14 days.